# Databricks Notebook: 01_extract_api_data.ipynb

# Este notebook extrai dados de países da API REST Countries e os salva no DBFS.

In [0]:
import json
import logging
import os
from typing import Dict, List, Optional, Any
import requests
from urllib.parse import urljoin

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"

)
logger = logging.getLogger(__name__)

# Constants
API_BASE_URL = "https://restcountries.com/v3.1/"
API_ENDPOINT = "all?fields=name,cca2,region,subregion,population,area,capital,currencies,languages,latlng"
REQUIRED_FIELDS = [
    "name",         # Country name
    "cca2",         # 2-letter country code
    "region",       # Continent
    "subregion",    # Sub-continent
    "population",   # Population
    "area",         # Area in km^2
    "capital",      # Capital city
    "currencies",   # Currencies (dictionary)
    "languages",    # Languages (dictionary)
    "latlng",       # Latitude and Longitude
    "timezones",    # Timezones
    "flags",        # Flags (dictionary)
    "borders",      # Bordering countries (list of cca3 codes)
]

# Define o caminho de saída no DBFS
# Usaremos /FileStore para facilitar o acesso via UI do Databricks
output_dir = "/Volumes/workspace/default/data"
output_file_path = os.path.join(output_dir, "countries_data.json")

def fetch_all_countries_data(base_url: str, endpoint: str) -> Optional[List[Dict[str, Any]]]:
    """
    Fetches all country data from the specified REST Countries API endpoint.

    Args:
        base_url (str): The base URL of the API.
        endpoint (str): The API endpoint to fetch data from (e.g., "all").

    Returns:
        Optional[List[Dict[str, Any]]]: A list of dictionaries containing country data,
                                         or None if an error occurs.
    """
    url = urljoin(base_url, endpoint)
    logger.info(f"Attempting to fetch data from: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        logger.info("Successfully fetched data.")
        return response.json()
    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching data from API: {e}")
        return None

def filter_and_standardize_data(data: List[Dict[str, Any]], required_fields: List[str]) -> List[Dict[str, Any]]:
    """
    Filters and standardizes the raw country data, keeping only required fields.

    Args:
        data (List[Dict[str, Any]]): The raw list of country dictionaries.
        required_fields (List[str]): A list of field names to keep.

    Returns:
        List[Dict[str, Any]]: A list of dictionaries with filtered and standardized data.
    """
    standardized_data = []
    for country in data:
        filtered_country = {}
        for field in required_fields:
            if field == "name":
                filtered_country["name"] = country.get("name", {}).get("common", "N/A")
            elif field == "capital":
                filtered_country["capital"] = country.get("capital", ["N/A"])[0] if country.get("capital") else "N/A"
            elif field == "currencies":
                # Extract currency code and name, e.g., {"BRL": {"name": "Brazilian real", "symbol": "R$"}}
                currencies = country.get("currencies", {})
                if currencies:
                    # Take the first currency found
                    currency_code = list(currencies.keys())[0]
                    currency_info = currencies[currency_code]
                    filtered_country["currency_code"] = currency_code
                    filtered_country["currency_name"] = currency_info.get("name", "N/A")
                else:
                    filtered_country["currency_code"] = "N/A"
                    filtered_country["currency_name"] = "N/A"
            elif field == "languages":
                # Extract language names, e.g., {"por": "Portuguese"}
                languages = country.get("languages", {})
                filtered_country["languages"] = list(languages.values()) if languages else []
            elif field == "flags":
                filtered_country["flag_png"] = country.get("flags", {}).get("png", "N/A")
                filtered_country["flag_svg"] = country.get("flags", {}).get("svg", "N/A")
            else:
                filtered_country[field] = country.get(field, "N/A")
        standardized_data.append(filtered_country)
    return standardized_data

def save_data_to_json(data: List[Dict[str, Any]], output_path: str) -> None:
    """
    Saves the processed data to a JSON file on DBFS.

    Args:
        data (List[Dict[str, Any]]): The list of dictionaries to save.
        output_path (str): The full path to the output JSON file on DBFS.
    """
    # Databricks automatically handles DBFS paths with standard file operations
    # Ensure the directory exists
    dbfs_output_dir = os.path.dirname(output_path)
    dbutils.fs.mkdirs(os.path.dirname(output_path))
    
    try:
        with open("/dbfs" + output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        logger.info(f"Data successfully saved to {output_path} on DBFS.")
    except IOError as e:
        logger.error(f"Error saving data to file {output_path} on DBFS: {e}")

raw_data = fetch_all_countries_data(API_BASE_URL, API_ENDPOINT)
if raw_data:
    processed_data = filter_and_standardize_data(raw_data, REQUIRED_FIELDS)
    save_data_to_json(processed_data, output_file_path)
    # Opcional: Retornar o caminho de saída para o notebook orquestrador
    dbutils.notebook.exit(output_file_path)
else:
    logger.error("No raw data fetched. Exiting.")
    dbutils.notebook.exit("Failed")
